In [ ]:
import pandas as pd

from commodity_factors.data import prepare_kalman_data

df = pd.read_parquet(
    "../data/processed/wti_panel_2021_2025.parquet"
)

dates, Y, tau_real, dt = prepare_kalman_data(df)

print("Prices:", Y.shape)
print("Maturities:", tau_real.shape)
print("Date intervals:", dt.shape)

print("First date:", dates[0])
print("Last date:", dates[-1])

print("First day's log prices:", Y[0])
print("First day's maturities:", tau_real[0])

In [ ]:
from commodity_factors.kalman import kalman_filter
import numpy as np

params_initial = {
    "kappa": 1.5,
    "sigma_chi": 0.30,
    "sigma_xi": 0.20,
    "rho": 0.30,
    "mu_xi": 0.02,
    "lambda_chi": 0.01,
    "lambda_xi": 0.01,
}

results_real = kalman_filter(
    log_prices=Y,
    tau=tau_real,
    params=params_initial,
    dt=dt,
)

real_states = results_real["filtered_states"]

print("States shape:", real_states.shape)
print("Log-likelihood:", results_real["log_likelihood"])
print("All finite:", np.isfinite(real_states).all())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(dates, real_states[:, 0])
axes[0].set_ylabel("Short-term factor (chi)")

axes[1].plot(dates, real_states[:, 1])
axes[1].set_ylabel("Long-term factor (xi)")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
from commodity_factors.estimation import estimate_real_parameters

fit_real = estimate_real_parameters(
    Y=Y,
    tau=tau_real,
    dt=dt,
)

print("Success:", fit_real["success"])
print("Message:", fit_real["message"])
print("Log-likelihood:", fit_real["log_likelihood"])

In [ ]:
import pandas as pd

estimated_params_real = fit_real["params"]

parameter_table = pd.DataFrame({
    "Parameter": [
        "kappa",
        "sigma_chi",
        "sigma_xi",
        "rho",
        "mu_xi",
        "lambda_chi",
        "lambda_xi",
        "delta",
        "sigma_epsilon",
    ],
    "Estimate": [
        estimated_params_real["kappa"],
        estimated_params_real["sigma_chi"],
        estimated_params_real["sigma_xi"],
        estimated_params_real["rho"],
        estimated_params_real["mu_xi"],
        estimated_params_real["lambda_chi"],
        estimated_params_real["lambda_xi"],
        fit_real["delta"],
        fit_real["sigma_epsilon"],
    ],
})

display(parameter_table)

In [ ]:
results_fitted = kalman_filter(
    log_prices=Y,
    tau=tau_real,
    params=estimated_params_real,
    dt=dt,
    sigma_epsilon=fit_real["sigma_epsilon"],
)

fitted_states = results_fitted["filtered_states"]

print("Fitted states:", fitted_states.shape)
print("Fitted log-likelihood:", results_fitted["log_likelihood"])

In [ ]:
print(
    "Initial log-likelihood:",
    results_real["log_likelihood"],
)

print(
    "MLE log-likelihood:",
    results_fitted["log_likelihood"],
)

fitted_log_prices = results_fitted["log_likelihood"]

In [ ]:
Y = np.asarray(Y, dtype=float)
fitted_log_prices = np.asarray(fitted_log_prices, dtype=float)

rmse = np.sqrt(
    np.mean((Y - fitted_log_prices)**2, axis=0)
)

for j, error in enumerate(rmse, start=1):
    print(f"Maturity {j}: RMSE = {error:.6f}")

In [ ]:
innovations = results_fitted["innovations"]

prediction_rmse = np.sqrt(
    np.mean(innovations**2, axis=0)
)

comparison = pd.DataFrame({
    "Maturity": range(1, 7),
    "Prediction RMSE": prediction_rmse,
    "Fitted RMSE": rmse,
})

display(comparison)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

selected_dates = [
    "2021-06-01",
    "2022-06-01",
    "2025-06-02",
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, target_date in zip(axes, selected_dates):
    t = np.argmin(
        np.abs(dates - pd.Timestamp(target_date))
    )

    ax.plot(
        tau_real[t],
        np.exp(Y[t]),
        "o-",
        label="Observed",
    )

    ax.plot(
        tau_real[t],
        np.exp(fitted_log_prices[t]),
        "s--",
        label="Schwartz-Smith",
    )

    ax.set_title(str(dates[t].date()))
    ax.set_xlabel("Time to maturity (years)")
    ax.set_ylabel("WTI futures price ($/barrel)")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Converged:", fit_real["success"])
print("Message:", fit_real["message"])
display(parameter_table)

In [ ]:
from commodity_factors.evaluation import evaluate_out_of_sample

evaluation = evaluate_out_of_sample(
    Y=Y,
    tau=tau_real,
    dt=dt,
    dates=dates,
    test_start="2025-01-01",
)

print("Training ends:", evaluation["train_end"])
print("Test observations:", len(evaluation["test_dates"]))
print("Overall test RMSE:", evaluation["overall_rmse"])

In [ ]:
import pandas as pd

test_results = pd.DataFrame({
    "Maturity": range(1, 7),
    "Test RMSE": evaluation["rmse"],
    "Test MAE": evaluation["mae"],
})

display(test_results)

In [ ]:
import matplotlib.pyplot as plt

n_test = len(evaluation["test_dates"])

observed = Y[-n_test:, 0]

predicted = (
    observed - evaluation["test_errors"][:, 0]
)

plt.figure(figsize=(12, 5))

plt.plot(
    evaluation["test_dates"],
    np.exp(observed),
    label="Observed",
)

plt.plot(
    evaluation["test_dates"],
    np.exp(predicted),
    label="One-step-ahead prediction",
    alpha=0.8,
)

plt.xlabel("Date")
plt.ylabel("WTI futures price ($/barrel)")
plt.title("Out-of-sample predictions: 2025")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

benchmark = df.copy()
benchmark["date_"] = pd.to_datetime(benchmark["date_"])

benchmark = benchmark.sort_values(["futcode", "date_"])

benchmark["previous_log_price"] = (
    benchmark.groupby("futcode")["log_settlement"].shift(1)
)

benchmark["previous_date"] = (
    benchmark.groupby("futcode")["date_"].shift(1)
)

benchmark["error"] = (
    benchmark["log_settlement"]
    - benchmark["previous_log_price"]
)

# Restrict to 2025 and valid consecutive observations.
# This also excludes observations with no previous price.
test_benchmark = benchmark[
    (benchmark["date_"] >= "2025-01-01")
    & benchmark["previous_log_price"].notna()
].copy()

benchmark_rmse = (
    test_benchmark
    .groupby("maturity_rank")["error"]
    .apply(lambda x: np.sqrt(np.mean(x**2)))
)

display(benchmark_rmse)

In [ ]:
ss_errors = pd.DataFrame(
    evaluation["test_errors"],
    index=evaluation["test_dates"],
    columns=range(1, 7),
)

ss_errors = (
    ss_errors
    .stack()
    .rename("ss_error")
    .reset_index()
)

ss_errors.columns = ["date_", "maturity_rank", "ss_error"]

comparison = ss_errors.merge(
    test_benchmark[
        ["date_", "maturity_rank", "error"]
    ],
    on=["date_", "maturity_rank"],
    how="inner",
)

comparison = comparison.rename(
    columns={"error": "rw_error"}
)

results = comparison.groupby("maturity_rank").agg(
    ss_rmse=("ss_error", lambda x: np.sqrt(np.mean(x**2))),
    rw_rmse=("rw_error", lambda x: np.sqrt(np.mean(x**2))),
    n=("ss_error", "size"),
)

results["improvement_pct"] = (
    1 - results["ss_rmse"] / results["rw_rmse"]
) * 100

display(results)

In [ ]:
from commodity_factors.one_factor import (
    one_factor_kalman_filter,
    estimate_one_factor,
)

train_mask = dates < pd.Timestamp("2025-01-01")
n_train = train_mask.sum()

Y_train = Y[:n_train]
tau_train = tau_real[:n_train]
dt_train = dt[:n_train - 1]

fit_one = estimate_one_factor(
    Y_train,
    tau_train,
    dt_train,
)

print("Converged:", fit_one["success"])
print("Log-likelihood:", fit_one["log_likelihood"])
print("Parameters:", fit_one["params"])

In [ ]:
results_one = one_factor_kalman_filter(
    Y=Y,
    tau=tau_real,
    dt=dt,
    params=fit_one["params"],
)

one_test_errors = results_one["innovations"][n_train:]

one_rmse = np.sqrt(
    np.mean(one_test_errors**2, axis=0)
)

In [ ]:
comparison_models = pd.DataFrame({
    "Maturity": range(1, 7),
    "One-factor RMSE": one_rmse,
    "Two-factor RMSE": evaluation["rmse"],
})

comparison_models["Two-factor improvement (%)"] = (
    1
    - comparison_models["Two-factor RMSE"]
    / comparison_models["One-factor RMSE"]
) * 100

display(comparison_models)

In [ ]:
print(
    "One-factor training LL:",
    fit_one["log_likelihood"],
)

print(
    "Two-factor training LL:",
    evaluation["fit"]["log_likelihood"],
)

In [ ]:
from commodity_factors.schwartz_smith import futures_log_price

params = evaluation["params"]

states = np.asarray(
    evaluation["kalman_results"]["filtered_states"],
    dtype=float,
)

fitted_log_prices = np.stack([
    np.asarray(
        futures_log_price(states[t], tau_real[t], params),
        dtype=float,
    )
    for t in range(len(Y))
])

print(fitted_log_prices.shape)
print(fitted_log_prices.dtype)

In [ ]:
rmse = np.sqrt(
    np.mean((Y - fitted_log_prices)**2, axis=0)
)

for j, error in enumerate(rmse, start=1):
    print(f"Maturity {j}: RMSE = {error:.6f}")

In [ ]:
from commodity_factors.evaluation import forecast_h_steps

params = evaluation["params"]

states = np.asarray(
    evaluation["kalman_results"]["filtered_states"],
    dtype=float,
)

def evaluate_horizon(h):
    predictions = []
    actuals = []

    for t in range(len(Y) - h):
        target = t + h

        if dates[target] < pd.Timestamp("2025-01-01"):
            continue

        elapsed_years = np.sum(dt[t:target])

        y_pred = forecast_h_steps(
            filtered_state=states[t],
            target_tau=tau_real[target],
            params=params,
            elapsed_years=elapsed_years,
        )

        predictions.append(
            np.asarray(y_pred, dtype=float)
        )

        actuals.append(
            np.asarray(Y[target], dtype=float)
        )

    predictions = np.stack(predictions)
    actuals = np.stack(actuals)

    errors = actuals - predictions

    return np.sqrt(
        np.mean(errors**2, axis=0)
    )

In [ ]:
rmse_1 = evaluation["rmse"]
rmse_5 = evaluate_horizon(5)
rmse_20 = evaluate_horizon(20)

horizon_results = pd.DataFrame({
    "Maturity": range(1, 7),
    "1-day RMSE": rmse_1,
    "5-day RMSE": rmse_5,
    "20-day RMSE": rmse_20,
})

display(horizon_results)